# 信息熵与基尼指数怎么选？

**面试回答：**熵和 Gini 都度量节点类别混杂度，树比较的是加权子节点不纯度下降。二分类下常给出相似排序，实际更重要的是样本量、类别权重和验证。

## 真实案例

客服优先级树按投诉次数切分，目标是区分是否需要升级。

In [1]:
import numpy as np  # 导入 NumPy 手写不纯度。
ticket=np.array(['C01','C02','C03','C04','C05','C06','C07','C08'])  # 构造客服工单编号。
complaint=np.array([0,0,1,1,2,2,3,4])  # 记录历史投诉次数。
y=np.array([0,0,0,1,1,1,1,1])  # 标记是否升级。
print('工单 | 投诉次数 | 升级')  # 输出工单表头。
for n,v,c in zip(ticket,complaint,y):  # 逐条展示样本。
    print(n,v,c)  # 输出一条工单。

工单 | 投诉次数 | 升级
C01 0 0
C02 0 0
C03 1 0
C04 1 1
C05 2 1
C06 2 1
C07 3 1
C08 4 1


## Baseline / 基线

基线不切分，所有工单都预测父节点多数类。

In [2]:
baseline=int(y.mean()>=.5)  # 计算父节点多数类。
baseline_acc=float(np.mean(np.full(len(y),baseline)==y))  # 计算不切分准确率。
print('不切分基线类别=',baseline,'准确率=',baseline_acc)  # 输出基线。

不切分基线类别= 1 准确率= 0.625


In [3]:
def entropy(target):  # 定义二分类信息熵。
    p=target.mean() if len(target) else 0.0  # 计算正类比例。
    q=np.array([p,1-p])  # 组成两类概率。
    q=q[q>0]  # 移除零概率避免 log 零。
    return float(-(q*np.log2(q)).sum())  # 返回熵。
def gini(target):  # 定义二分类 Gini。
    p=target.mean() if len(target) else 0.0  # 计算正类比例。
    return float(2*p*(1-p))  # 返回 Gini 不纯度。
def weighted(metric,threshold):  # 定义候选切分后的加权不纯度。
    left=complaint<=threshold  # 标识左子节点。
    return left.mean()*metric(y[left])+(~left).mean()*metric(y[~left])  # 返回加权子节点指标。
parent_entropy=entropy(y)  # 计算父节点熵。
parent_gini=gini(y)  # 计算父节点 Gini。
print('父节点熵/Gini:',round(parent_entropy,3),round(parent_gini,3))  # 输出根节点混杂度。

父节点熵/Gini: 0.954 0.469


In [4]:
rows=[]  # 创建候选阈值结果列表。
for threshold in [0,1,2,3]:  # 枚举投诉次数切分阈值。
    rows.append((threshold,parent_entropy-weighted(entropy,threshold),parent_gini-weighted(gini,threshold)))  # 保存熵增益和 Gini 增益。
best_entropy=max(rows,key=lambda row:row[1])  # 选择熵增益最大的阈值。
best_gini=max(rows,key=lambda row:row[2])  # 选择 Gini 增益最大的阈值。
print('阈值 | 熵增益 | Gini增益')  # 输出比较表头。
for row in rows:  # 逐条输出候选结果。
    print(row[0],round(row[1],3),round(row[2],3))  # 输出一个切分候选。
print('最佳熵/Gini阈值:',best_entropy[0],best_gini[0])  # 输出两种准则的选择。

阈值 | 熵增益 | Gini增益
0 0.467 0.26
1 0.549 0.281
2 0.204 0.094
3 0.092 0.04
最佳熵/Gini阈值: 1 1


## 结果解读

必须按左右子节点样本量加权；只看一个很纯的小叶子会选择偶然规则。熵和 Gini 都不能表达漏掉高优先级工单的业务成本。

In [5]:
threshold=best_gini[0]  # 使用 Gini 最优阈值作为教学树桩。
pred=(complaint>threshold).astype(int)  # 将右子节点预测为升级。
acc=float(np.mean(pred==y))  # 计算树桩准确率。
print('树桩阈值=',threshold,'准确率=',acc)  # 输出最终树桩结果。
print('生产差距：需加入类别权重、最小叶子、误报成本和时间外验证。')  # 说明工程要求。

树桩阈值= 1 准确率= 0.875
生产差距：需加入类别权重、最小叶子、误报成本和时间外验证。


## 失败案例与修复

错误做法是未加权地平均左右不纯度，使一条工单的小叶子被过度重视；修复是使用样本数加权。

In [6]:
bad_threshold=3  # 选择会产生单样本右叶的阈值。
left=complaint<=bad_threshold  # 构造该失败切分左右集合。
unweighted=(gini(y[left])+gini(y[~left]))/2  # 错误地不按样本量平均不纯度。
correct=weighted(gini,bad_threshold)  # 正确计算加权不纯度。
print('失败：未加权Gini=',round(unweighted,3))  # 输出失真分数。
print('修复：加权Gini=',round(correct,3))  # 输出正确分数。
print('高基数特征还需增益率或候选限制，不能只换一种不纯度。')  # 说明进一步约束。

失败：未加权Gini= 0.245
修复：加权Gini= 0.429
高基数特征还需增益率或候选限制，不能只换一种不纯度。


In [7]:
assert len(ticket)>=5  # 保护案例样本数。
assert best_entropy[1]>0  # 保护熵切分带来增益。
assert best_gini[2]>0  # 保护 Gini 切分带来增益。
assert correct!=unweighted  # 保护加权修复确实改变数值。